In [ ]:
# ATLAS v3.2.5 — UNA SOLA CELDA | FULL O3 | SIN SUBMISSION
from pathlib import Path
import base64, zlib, hashlib, torch

DATA = Path('/kaggle/input/competitions/hackathon-ia-pucp-2026')
CODE = Path('/kaggle/working/atlas325_code')
OUT = Path('/kaggle/working/atlas325-full')
CODE.mkdir(parents=True, exist_ok=True)
required = ['train.npz','test.npz','static.npz','sample_submission.csv']
if not all((DATA/n).is_file() for n in required):
    candidates = [p.parent for p in Path('/kaggle/input').rglob('train.npz')
                  if all((p.parent/n).is_file() for n in required)]
    if len(candidates) != 1:
        raise RuntimeError(f'Indica DATA manualmente; candidatos: {candidates}')
    DATA = candidates[0]
# Solo se exige T4 x2 si hay que entrenar; recuperar CSV no necesita GPU.

def sha(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(2**20), b''): h.update(b)
    return h.hexdigest()

# Fuentes originales comprimidas, incluidas en el notebook (sin Internet).
PAYLOAD = {'Atlas_v3_2.py': '...', 'Atlas_v3_2_5.py': 'eNqlXety20ay/u+n4GYrB4A0gkhK8oXMpNZJnJzUOrHLVnbjYrFQIACSsEAAAkBRlMvvfr7umcGFhOzsnmxCAXPt6en79GC/+eabl1Xil4O7C3fsXg1yvwrWUekO3kW327iIysE23WRhvIyjcMAtvbsLb+zm+8EiKuMwGlTruBws4yRyv/nmmyfxJs+KapCVT5ZZMbgZxOnAtt68ffX7D69fvvd+/+M37/p/3716+dN7S1hvfnt7UPLbP193SpwJRnKj9C4usnR2M5fWpWWm8ItV7hdlJAYfyywVg41frcWgijcoWfvlOokXYhBk+V4M8sSvAM7mybLINrREqhzocd7i1YyZbjdYGLCR5qYo99MQBfg3D01ZlRXBWg3Gj2aoNFWFZRDnezeJUz9ZmbpgnXlLP0B7wc9lltxFuvlNEvlF6m6rOMFq7ytaiulXYP5sEz9EoVfe1RA0O0GQ3T35/c273+SdS3+mg3+9fPceL/TnyZt3v/7y6+/v5Sfrzcia2NZ4OL4cXoyeDYdAt3rBf/XL5XiEFwd7M65bX46OGuiXq7r1Rd36qtvgefPybMitPz95++7Vu1e/SGsRLF6EVy/CYBiNLp5fvViEz4Kn/ovw2fKpv3h6sQwuo8gfL69Gz66Wzy+evxgvnl+Nhy/GT/3l8mp58cx68uOb33/+9Rcs7p2/IwgK/BE/+wmogn8B2tvED6JFRrW5fjxssd6XcVByC/140OJdFERp1WlwXWyb+p+KeFn19KdGWO+TMFpiz6PgxsuLrMqCLLGdyZMB/sklkZ/tecRAnue4u7hae6m/iWyr2WPvCh2jIlrFZVVk7ia0HO4dLwd3brn27dz5m1RYnYBg4jIa/MtPttGrosgK23pb9/WrOEsHm7jcEJtjFAatzJO4sq/FWmRFvIpTDVuwrUTgJyJKQ6nJaKYazKeDqpBp7i7BWGmWPkRFZl9/hw7OdBD4hzX29feS6v6H2vgJ/q6/vZBy6KB1dNfbmltdf4epO639soxA/UmU2lXhDMAa/Bz4zXN0p543/r19PauKuUNw8YKKqNoWKUDHskR0pxb/APEFptzbf4oKGLiJMeXEYJfe/iZrwpmYIe7c415S1tuvdqeQh80UvUwHfx+sojQq4gBy8x5yFUuHsIAMGCz8Mi4FCgZ+GMa0W34yyBZY9R3vXckj/33wHhiKSLiqAdTEaLpN44r7V6ipKlRV2WAdJWG2rQahX/nuYPwCtOinaZSUqKz8xOUxb2Vxjp2ACIps/PWLwt/bs/GVe/FUjN3h06sx/jx99ozfnj97Ohdhtc8jaS0vIQEusKrbmeu6YjQ/k+MXw3PVh4eGRJMzGttP17Zq9XHuDEhBfKQ1QMStIns0duanx81OuiX2x9OR8+1ofNz/ypm3dxndgiwN/CpK8Z89KwRKysoPbmzAI/z7uJRnI2deP7l+SQuyeUGKNnb+3cr2xc6ZtEbFgGVWhFll74SP3lEp7aEAeZ7v3HK7sR3F7km8oXl9bPtOhP5ehCQkNGn9HXuxx5gg+/guoj0Cq0OvgHfTOF2xFgOsm1zQ+nhTB9lycDGkfqXaMJBKJGdgxTLJcgAxU8snrKyzbdEgZnypZ2Wivj/kt7WU1B4biLr4nsie+Ci+dwYgkWjAxMADUfGa2HAXy90svsfcoQRA6nEjFbroTexiNAs3NJd/FxU+eodiF8WrdVVKVDbwLAeMmAZCXpvsIjo+scOzcOMINXoH6bHGerf/4txuza16n5yM2zCcuuOrphutdcLzElZKL4lvInvT1BO6XT/PIZLszdniBONNFeZN6cI5oD+/VFxEXTvc0qpTQ7RrFemBV6MQMrmIlg312XnoVplHdUQhqCarxq+k9e2Hbzffht/+r+Wcoc21IR+bujsus7lXRmCIsLTp3WNLx3bOnz+Fcj+/GGpNEPkliRw7SESZgHTDZvIgma3npyX9Ek+CwHPs6Rl0uri4xAiOC2tx7YOD7LORcE5t/JxgIDcN4w34Sy8MYsnzU1hAhRaKH8S1gEgulRBtVGjBSlcqTco0op4b6UxSTrViiU+v3I7fDkV3vZFlBbncA0UDQ0uQT8tqZiXRyg/21lzWcnHkTDVWSqVaHmSjR0jpiIp/1/zLOmU6WK8lv0+rSnL1dJBKrcow3FJCQR/tMNrNzkbzec9OO4r9mE54HtpsMCeBGd3nYzt0CEsaQ4aXszQq7dSZ7uQOAmsT+amNl6XcdeUfGwBEBNJIsgexhihbilBLsunDmTykFyKYzjiQTiHBA20cb8CmJINvi6rNm1RLUDycPChZDMIZO456HjYMi6JRdHaJeaUhv4fzMuxMB3IcCygSJQkh5ORDTZUpKp1prsUUVWI1ZK8AO7CPQklFZ/lmOvCEJ/JAdi1v27Q7MWvYLWcT8TvwOQdHe0EGyzxFm1Ju4hRCV6SYT6hBPHBjFcnRM8y3kmakf+SBez1dlV9B0Opk9TgyVudypYyCfSD2Zb1bHxT9dXdssN9LW1Wd1VvH/dTWOefkQDgdlIlLhU3CV5KlDCuqV0Uc2uXM2hVLD1WlNRfmDVYK3kDz0T10mbTij6CDVZFtc16pvyht9HALrA92sMHhGbqvlqUeDFRUrAiPI+dkdHFqumXpo9141la3J5rXwziobMXBDf8OHcXpdQGzKDFQU0TcJAKif+ICUDcouQxBGL5H9CrzDT+39j0PuATtVqXYB1LtCJCrF6/+OLWaXrGSxoRQ8LdbbLSqb0SVD0kVwstegTvESlQVRhJasJGr40w327aK9Y/pZBp+jbxs/2yzZd3YQ2Tu8MqZskEPp2W7ST1lQc0aQSJU//OwnIO7js0LvXYsnmi/8DfSd6//Ye/MBp74JPxW3G2bJGxkUDOa+/mJNqm4yWw4lyD6p1Oq5jlmk0nd/HQ0P5VoNV1ElS9rD9tu/G497MH0+z0eAfe8KyicRvNzIEPc017NbGuztQTWK2yrDC0RlvREU1qCfp35BBpjaX26+ex9WhWfoTTu21YB1AVrQbhzRJhGB5WVUGroutFucVrZbe3jTGrnQ/Xy9CB1bwUyEzapEupML+g6rRWFMDUgb1S0pand5wo1coI6BQnxOR5KelAig18xiZG9Xfm/gsBpZDDt1gNJaYyLboaXAMg/zHvDT9bcvVbDK66y5kaKUJl6RFnO+m6TV3vbjD++vBpCbnWsqv+a6Y547gvsQPP7DvMEk8Fmq6mA17EEnvT713kl12Qpbf8f3JfIy4zWQajvCPqDxo4R1jWl8MbmdXOI82cw167Q7oRE/Wlnd/eB3t19aXZXUWuWwxhBI18sxF6jKpT2otnq5zA8faU+poW09/015PSuWAwTV4cnoVb4Q8Eqf9py3Qjuul3RbXfeEmgjGPEnalzSiKA/MRTulWOCLllWRp6WaAp+QZxQiju/iP1FYsxJ0meN7J/RGMJ9fiVGLnEPnC7ZkAwPAGmWZGUZlbXLYNuXwCtZFNmW1MGwpjly9kB0NMxsNJk3tFYVzJfld6iaRnfqRZJPBfusxjqr8gX/7jmiMe3IpliQsBhEMOkh0Ssi6ZjsdgXeLIZcvCNElvadiwXDU/Fn0R2NSL+7E6i5PT06ELD1wGoNp5KkUHSnpfAJMZaRT0RztCLnu8vJQEnYspbil7QHU/DyBiaDe/Wk9rDqCQp2DlkZKUDP1ZSaVHTXZZL5FS9oxpvDyr3YKIsRUhq4MFO7VydH7bRKc+anqFVDctBM731tFGuYVYsOC3VJ/0RPJljuahjVH2Hg0H9hxicQp7YjIKZBNWkVwwdjSqoxZ6jUD9YRBGHlE2n5HkWvyR25FousMkE5sclCQ6waNgZC1cpWI0k/IthWh1E77K/gEIMH7mEV0QTJHF3DkQfJZc0KAH6xr6sIqFYlgSwbuLcJBvIgbsZXT6WOw7vqFfT3x+vrdy+992/+ePfjKzdKA0AKunLX0X0YryKFLgpNmgFUeLMOj4ooz4J1KWfjK3HxdE57ySuGjMcOWmo/Sa9rzELJwdBOg8jeig5qOQAortnxa6FOZFsToqFVEoUW3owbz6dpIHWwcbphG+Ovb9uUzhwkBj+3apA87u+m+YMFL9Mv4BQCwW4ewYZmVogKeGbruJLUGR4d0F3adeAXFQ0zPQBRReSHKYQ9GpPbmpFLQycjLlgIXj9t94ObZzlMGIBMEMOFhDZTIVXdQVId+8+qYGbVBGPNvzNloC1rrnxuLTE9kDi4RE/AlAJvTNMKli8fekQArPMtbIeKLX72gkMq+OnVv3798dV7a96WR81O50V2v7cmufxk/Y65s5012ZI5s+JgKQYKocNoH1D6bwzCIMATV3+h1ze+dC/hywR+IknEM0++GDqfu7GgTkApJ8CKLSv8VcTQ9o+vdx11P7/87dfXv9I6SJlb72oA5X8O7tMrA++zGlwaVCPgvxmyDwP1kkkywz94YGoShl4UOYXbTV7aVCbKrKi8mwid+aRDdCSEODnJ1YiGxKLKzh0p7XrjPh9tq+JgagnY//nqw/vxM7JSmdZ8yOh7lw0ZDEKsSPM4tUVjCDIul3EaQw/ewxZFHxUpZvs9d+/oXARspCDjgJQdhyW5q20j8IjcDiyqBvUz6jk/hWqN/bJbgXLxAdKDn2gDzKMzU5uh+uK9nndHO4ltiRfqrAab+elmcs/d2o4IVllF0OjO5/YMeu732DUoG6s9o7GtMLyf58me8KqHTh8feyc6gGqkUQCNSAYS5RozYPA4vKdQNDRdOwytzZ6W4VQ5s3Hb/llCuh3YwGFFxpAzxUYdV7FpxNOZoCuaOTSveV/aGFNQaS0p2xqimXq3E5vNI9iepMGXsN2PaKChZhuW9EoTtoZnWd+41ztBJCM3m47BsbSbgHsaqFhxE35nine4tH26AYQclQEpjrZTWFGRJsF/QoeB+7TNGesh0dG9rE9aZgMxfkcrsKWhSlkvGKMmK4ooIJ/TJ23LKlEFe7PYnLC1NKqyfA/pallkD1EqNdf5hst83iG9Fwr3mqPYsfoaFdJUM0Axdx6xxIn2UD9rWjLhzduEZmCaofG8Cxhb6gycDgLrJwIyUB0AZ3fXeaF9W6iwCfwWcVBSSQgX5tromcZeUSbEQfj40AAc9cWRp3TA+5WO496OrDfQk7p5ikgVBWFEcw4RRgkbSGgJe93F5i55Bj2O1TqN0O76H89p6LaQAWeVLedMCQJHOSXsk6iNNj4WSbtsRydi010U3SR7egr9WD1AJZSl/PS5JhA6bRc58zmwa1i9Zaa059aeVD5jGMghUw/kPPGZRO1TKJ/wUmmfUKQ0wQOdmQBCkZbkujAoM5p/LsuynpAt12xno1kd9HnE1oeMJOjLmzhJtLMECOmNezsUQgBAHp3wlezMpQ4UM49GwMAu+e39K886/dgcp1LeyFwY56w8xzCtaBhh1vAAA1nyPgwhf0rtJjoH7mnadU/TtqI1RzkT3h8zMMNHVCXZZ47nZKLI+HQE2NWkVJYeTdUceg45JoKpxLOD2VKZovckPn027zk5ZFgU1XSAoSIZn58/68DQjELrN+DURFXomCE4Pg5ZBXiakS1BeKTYoZrLKyEvYTML9UoVjJC6nN+c+QRc9RNMsZ8LShEpmHmC8o4VztL6RLN+dlFgqfi7PjF70rHpWDnp0UvlgCgLkClTBy/SEstk+qxNB4JYRy+zLIlCb0W7CSGfFTCgAlhftMhIkKI1QSI0XkFT2MxUnSAbIe2ezGLtDlkGj421pjpNF7ptPcOjLVMzqlqEHvK4nRG6HX5ZYAPPzItPu6mzOPQKZ5TlBC9QFs34U5/eWmsAsEUL0Okqy0KZfj+ssZFDPMJksG9rzj6AYnbbEJR5dlqA9dc35uwuTktYvpNb2cLD98NWbI16mOHaWMVootUHr873Tbv2KntaOuK2xU07kaZy4+c2igTJmRNbw9XshUZrY5HDGQVp6gMbv6w8ohzJ5DOD8hQJaRnij1LVGFSWSQwHf3TJAX2H2pXV6LKvzVm3kQeYvGXhB8SZWnoymAVJ83PzOMLcNcUbiDj4cbwE07Bv5PMUslfJbuifm4la2b3OZNGmphnSGJyMEzoEAOt6AN3LyMZqdVdyu7czCbOPf5M3ZoZWk88mgqffne9HSnG19OJrTt686AjQIuWTGnWq6YLefFi5Hkrt8XD89PS1Mx38IEdD/DP1/VZk9AeOjC4W0vddysm0waxpU/9DV4730kdXiv/ElmOLCJUTYgMUF2ouWkVFCSXw09lrCOwyfogkYKC8SjeI4sT+6fw1doszivgY6LQxs1FRx8t/oKOK2URMfgKr+6fygA0wo2LAEa2Nqlvc1K5M09M2O7brOjEcLQMWsxQyAwouVQ8t9vcPq5g+ltYiyypK8Ms/vaZDJ2X3xy+uCMe3Wz/lvK4HMXOH4yvhvnh2NXeaEF4ZaRp9gH0GhRdmSzLWRBHl4Bu2zPXJClQi/GEPHll2FxWJn3uLJAtuSha8hztyfv76S+zOK2fqDvKtRwcuHB3BShJbEb0RAiTA2LvXxR05cFh5wNeopvhJa7YiuoujnRcl4MRFEpkpSW2pyFdUc7YjJb1pa9hppmlD3J1cSx6atgVWS3504e2RFxpilcUIHu8TAA0a9WAU6OijBIAKKuBY2vfDA7buuiDo/eRJgCWUgx/hIHLyIby3NHV/y8JtfVBCOsbzKMDieXYZJUtRis1WlKHYxSF8vMD0bVvQ25zcS7fu50ypp5tGZOWzHeulERm87ZE4pqQa1oPK+mmaJaE0o7jLbRlhiVOMIgExFnA3Dm00gTDwTLrj6aWgEiy0LhIj5YRTOT0M4pLziFQMuY4VxNVaZXq7aeatCj88FEeY11WOPUs0WmFTRALkAJQ5S0GPIVSNjszQPpAmNCgVtrrTa9P5ECMSPZpQV1bs/CJU23YvKuG3lnErKRXAR+edp1Lv7NmzoRi/GAr8Dxr+vFNHpzSdo16ITezyZluR8X3BJ3WQMPa9y26dM72Vt7Ss7n4qpaNQ28q7u3U6zlCcfowgzzZMh8IvVmVtPdmqL4SUPaMKOuO4pXn1i559LkaOaAbFLoRJdEQ+rkqTjgpPY4qO2OE7ZTe2AsFpxQv2BgQzig2MtvQYZf8n+4maCiNvIDFt7c6r8EnQYjM6cv8g4nuhzvZ1KlrDTZDR8PzV8YcIIcGCSLMJB2jaMYC+YwSVJ8JK4M8ZZpmMrubqEA2bNRYXDqfwqlwtkwJsjlibDqQc6vacf4SnJzWH3Lmv3/z4z4nakY2fbjnpMCJ6i0LAwP5qW7Y8Jjl4SNWcf2k/1ZqdabTxJRE/jI8oZ1OCmzhuBJFok+pW91aYTT1bp1w/brco2BZ06CQvxkShDI2Ul88VcY6eMjj3vlQL80vDAoyZ+dlmOyPeFCqjRGeEnZfhcaneOGnWUh2PWXGC60E7/7idJpODhvv9MZAfGEjVXh2udlPmukMoNZlXehw+/nRfhv7m3wrPLixqOJkVmVewogvpDocjHdP0wiiAm04loKVgHYWdUZLC40LwcAEJXcZp9BLCEL5xunr9zkYjca1CjLWZphORz3l7nBNN/pQIAcUox9EZZSdtcg28y/EjaQXb0LcAgI+JDD42ufsLCILz5gtbNRFRSqfAoUStM93mKgY5nKpjjFYIMcqbwIKCoSU3o7Kf4xThEqPjDcYEYCHjVMlIDgaY5XWN34+HQQzK61AI6Oqcm6O95llmHycfT7n9ER0BxUo9sQqDZeNR7A2Uqc5vOqO3dJ6/rbIANKN50GM0t1Cu82xUYzYkR087uJ3kio/te392MxcV//r0250zkrnqbztn+z2qpxsZuSXs1wKis5ZXKlkkuYOVdGJvTiknzFHBKufUvTzZTCmhQCZ3Orv1ULHqHN8dpUZNuKl7cXWS3M3G81PYXfR0gafh6MSO2EMYT6BG4uXSDuMN7OIDgJzTnlbjo1bHYaZMXyRrDqwIGGg2vsDzM+EBnPE2gzujL/Jgq1RLzu2wukMqenf5jxrJXfjBDSt8R3ODu035wSNuI7dE3zmjULYSmGlWbLw+Vh+5jjK51EArEI+aqh4bijNX45rJmKOON+B4iO/JAOtSt1oSxIUat+bPUzk6atdrn6l30uSRH6w9zJh7ym2HCumsDXKMio8XjVWfkQXuvnhxJezRqQbBObdHw/rF6VxmALnrmUnEuOUecrfIUjigdlvCsglBYsYEGftOYyjJKC8oaNtvC8Cz5TEoQVwsk225bvGxAoIifrZOEqRwKi0dvotNgXhyo7jKgRo23FtkGx2NpyTHMjwuL0NHKWyp1DazUxdA2QVVtmwXqeWnMluOslma5l8b9muHWDyP125n7CbOfNGbIYzUN7vZTc8kq46PwMk8K0VH6ZpbckYMU5qFPi+HW0yH6wwzuYr1oR8cuGSvNolMsuBmRvmkc1ffxiADjMoo17Eue9xsoqa8B3QId0OOt25D0RBFzQSU19pqHp1e6yZHFpY2pKa50YAt7opTk7yy4cydyZc115+wD8cHWotzqSkY91F8PEXtNK9v1bCK6DGyyuT/YWSJYwsL432tkdrpnpaOZh+9OzruKg8P7lR6pTmBJqyrTNxa74SsMybjuU4T7OYXqOlPQ02OvIsZvJK4Iidf8OHCwe2Z8OD2Zp1xtLR0urA+HeCEowFnMOksNiiVlWQRqO7TqhMpk9m1/lJm2CNZZpTXdZS7pRKZvVbdnVvX1qf1nWSng1s77RSnttC16DpPndXEmjWmXOsBEWU3BYrSWzut4dj+jY+vJ0cXZ5fWS8acStCDUIAHlwKZUX2DdjL4RMB8bqni7r0ggP34haeTDrqnx6A1545fybxppR6wBDo5Kav2FSWVDLdNPbVH7Bzz9hLZbFukVHaJ6HGHkiL1fHWZhqKQinOuelKNu7kJ4wKgFHwvgu9K85Z62Y2WfiYHTx5dPp5+OfOOqMckC5G+pel7UohUusd0kVO6gVhk2ZLTmGQnI/CrqYB04B/S9fI66DtZ5J+nOcg/iCM6ReZsA/wpK7ozJkkCdc8Ale4uokBfwiHKVPvec9Jcyf+M1SGke24S6Pyr6iDxq8nGoyQ+deqc04t55kwHbXFoTPD0YFY6hbTzWRpMKLOBkoWPDA5j1qIH9qK+VK8RU1ZqWuuH2KfscppWP8smVSTvSRU5yKrRwoyDYHx0yfG7Ov3qKDFEW0Mx7LgVpwQEBWzj2jCoN8zQUhptC18FgvVxu+TPGzSZ0f3+Xnwvsgyqn7NVaArawYONU6TfBaDPggnKaeecF6N2TnotMwHc5TReRmV1fOY7jVP6PMcxfdB2CI5cCBVsWPPvNf06U6g5OVPaUGk85StP6DMPPcUjy5lPtWXH6FPPjVlA37cgzbgtSBa4yy1EUlSaj1tcr0mkv82y5NV9BF2SFR2DIvEXUdIEwuAy6gP1X9Pfs1/TfFtZ4vCbC2rZdID+a3rUhCVR0+LNm58fHSfLdIv+QVT1vwGaqWe9XlfOD/JHISpBffqYntf12ftEhu5nN68sXi290fru3PevXv30fn7sjh+jizK4vV1W3MBfgdPJnylB9bErtcTsVOOW28UGlHkUgRRNBNJvRx8PA4+zj3Nx+ZzN93JWX+MXBvgmzUMvA/RRFHK2pDO1bVLp03i+sbAsu2vMYXJ2LO8jUVYKJUSMdGHrDAaHGlTl8BBgcyVsGM9zkz5mLprmubkWwKLQtGr1YEHYAU0tsuiusCicY0TTmZm/l4ceBOHKLGc2uRjXa1IvRwtTxXp104JOrWhcio95/oKkpgqUqDitrS9IqkZneY7ZeIRWhnfPGN/RfbrL4yXAR4jo1isdOVqEPy3/ediWDphqsdrJlOHLFSk0GTPvyUlxkBVDOGTrkbLbug5EfS+DTqhmzTdU5i3FwjpD6fgZ1x3sAU8rrd/ie+uUupza1r/8wjq+9mH9yPc2LGe6g0hYZvLgshDbDLV2VGm2RBdz0clSVGVOc5/oKM6j4G5ExWQ3a+z+mvY79Kqu6cB4aVS12Dm13WEa6swknU/aWOUnJ7Qg5z/Ta38fvKfc9sFowKdODMJgW0JYk9NKAN7F2bYctK6zcLadipDCQqY3tzm3OU7o1KiabtWHK3LOpTde5pfTNZ3uvSl1XtUkbBadm1ND4fVtJmWttMwOejWbiuf5XF0Mo2KlpxV8Tb6n3hQaqeiOhNcdSRMsjD7fRM0OUpeLVpSIlt/6AsS2rOc6ciO51DnWieqzWm0t2Ci7jlJr6a+GAA84pp/81dyiEZAzANrCGL3VOOOq4xgy5ZgwO/418iemJYy+hWVpnR4wQ6cFcfRctHimwx294zzGK/wZlfa4He7hPFKTfPvpZqJMYJ2wc5Qz+lnJRdhS9NtK0Z2Sids+UTCCEL27IhCSNo8E3zbTtk47P108KpTIGqHAzdY01MZ6I0DagvuAlSLpdy9s7qdLeXC7c9+jDyHzuhqRUkedCS22oxIez1ul1Uq1ZiNAJR5EHvlFWadOMVsURZBFSx2S/zg3HwIQy4MCZzYUo7kJy3bsaBrkwJAm1yNRn1bqsaG7rgcFzuJAeR3wsA+rW1meWvvzqNy8ezMmpVsBMg0E3OtVVMkPzY2Zgk4joadhHcvH7vfVPq+ovVmpbyQMGs+4zymmtJYU4nhL3tbgbuw+swQHNzgCYnI/Z/qjJEI73UJ/VwvUZBxf8yCUWvGMf6X/itpBMW5Ox+kROz9JvK/HcClU9NhNQBOcYkuFeFkag3N6x5ESW1+3I1S6FD+hT8rQi9N1cbsZtZSVO5tZTKMQmtoEooiXB7xhbXabQI6cYJMqyxPp1H0+ATSfezNfLXRfFitwTVq9pTdyIKERwtDzdbFtnZ0RENifdQZ/C7uit1Ko0IawFtj49cYvbvDMgVk2h/lk22x736gcWmoant/4q1USncdKNfR0oIQjoY/VQx2+6WnF2GyD6xOUJzo62AKNyvtGUKRkCZ01afFtQw/EGvU2Vy4JWIvOHrGf9QSXz/uaB/n2TG1L72g1Mr/YaEu5HL1Vj6MPXHBGFtRfXlj9BYTWLhm/oEZvK1zrnGpDRPzXFgHdRYBQkbnLxEngQJ1ND79Z2BP1+2vhvo4lqgWTItlW8KsIWmMrVDvTuCQXSom8GjGsGf/7xU51DIkt8xq1NXz1NLW3QVcLGA4lRrutpiYG075ta2M1rkLIuVUHaVgKORzG9ioYinbjnZk2dIoTUm7pwXchVYjuUArqLx0edLU632m16m6tcHvH5jDGyKyj4Zpzgxa61PHB/NSeHQXi5l9AoHLX5g76La1P3UZfjYPwx1Ba6Mc4E401tS7g+pyAc6RscEELJducnA/GJe0It1JtG6+jbE4WaKQ2Eg5SAKHv+ZSPF8AcLyDMsp2Xx8ENLBelFCgIs5z8OZTLmfUnqO2aHtSV6LmxqQBVMzZHCQHESxLxEVGNNbf/HFI4nk4FuuB9YVsox+7xD8t0Tql7t2lya0Ilt6ddlBzutTOzFsr1Wc87A6t96lqX+gCtN6hIfVrH5eju3/lxQnagrb8/yAeotLSDME4nVKQw8wXK0ie4t+bwtpfS6rjQ8UVoFXtmvnrMuBNNhbw1py99kk+bZJy227jmx+dP1u/KZ68/C2ly2wa/vP1jkK/huVi9orWxDCatCVgUG3nTCtZKOW5gaccGhPWDGaiZ2ijewXhglP4AbXtiDZ3b1HQmp3Yn33pKxZJYpeKW2Gzkf9OsKy9n1opPr7sp2D24+xEoUtUG9tA65oH+u8GHMNf41CD1THeEqfAgPWdRyN6FHo7drx4WhbJJ+QS52V7eN6rTxtDc7OVCRfswXsm53zOra6jrrHVqVn9ZmAxIqT8j/MggLYPcOtZGrbyTLYyIkLJoMBuGjgPPT1aglmoNP5kNhJ1fpO10CLbnKVxU+V6RZZXCDr2ibs2fbExXUaFMd1XMfNFIRyo8VzeTtTgsDyvboh1ymdv2yubpB1P5ARK8V0Drekh4FtQqcVLf7mHrrL41YIit5i2//lbJrFUx7/AQOU+Enm54QDMc/tVTdYm2KoQnvJ4z1ZDsE5Jh+pCW3v/CKW37NJK6iK6Affxg8uA4nF+3wCDn//PFeN4RCo7AydpkN9FfOM01jnEnN/MYI7pRfUn1Pzn2Nl4c3/6gW1Y1ARHybg7vTbYuVunZP09JPsm+G5KdINOR9GGOs2uRo8O1GL0lqb8n34bpicRz61ZN8+EbdWy/3UA87FUA4SBkcHxYJwxW9V/BK6AfHZ+BEKjgvudRQF8rnvLXqbdVtiFeGvB5Ully0KklSrQcaSIUnh5LxynYI/rilndjFX8QzYCtbgYUtHg0tP2IFmzIsTCr7BWKU4059rU/8pm0p5R+J53Mu3wO8UdXnujs5V6LX1MLWXk2mrduPhUEKVrpkxGSnDctyqEbRu3adk/HObl83kDVKAsQCikD61rbBuxiDsCi/PlPHuLNxXQQ3QfJFpw70KlrgzqDi9FFEaBBC9H0kQqlyh+NoWhYulGWrodjvo6FXfhUm99dV8aZHMZzDpyWyZHL8rmPvEwUyAheLXM5KlSaiJA6uOa0P50U6t1FBZEsTUyGpzZOdSnboiLfV2tYcub/O8JV76anrW7H+XvQR5bwOZpgl0MtmT65pz7u7VUZ4cgxK85bZ5WUxROFVO0WqyRb2NaJpfibvmLDC7eVj5cz1v4mDzD92THBLOuXl9evLGbdTlAqptta1NfzwBKeR9Eoz7MmKir15P8Aj+VSOQ==', 'Atlas_v3_2_5_preregistro.md': 'eNplWE1vHDcSvetXFGAsIAOtni+NJGtiA15tEhvIrgVHMrBYBGtON9WiwiY7ZPd4ZOSQ/5AAe96jDzosdMt1/kl+yb4qdo9GimFbM91FsurVq1dFPaPXrVWRVrN8ms/pj19+oyb41hfeeiq8q7RVpSflWh2p1OQ6vYJ5vfnSBlOouLf3XlcmtsGT9YWyGZbrlcEKsor0jS66wmzuXX9ATv/whJ1gFIZ1et3q4Dw5Q9jbdiotKIyuVE5nm/vSVJ5uSYXi2qw8RU3GFba71Y7wt1bOXBkdW5/TG9Ns7uGpifRm8scvv755AZ8tsUfRh3xv79kzupjk9C5s7irt4MgtVarVe28RX8DhPtK/jDOF8RldGfc8o8uLswWOaYN2qjb4wP5z7K2q8U0LNMH4QMpSWso4AQqzDCmU3fMA6aMoEXf0VjWqVKc0PRyNZ7SPtTQ5hvvyPRP79I6fTUbjQ/glP3vbw+H5YMuf+2dztj3Bz2Q7ZVjk+2DLn/FsDNvj5zl9QFDKKUn29CSj2RgYzQ6p3NzhIbbQrlSIN6fvPIcT2pRO5DwiFAewkdTY6KJFsmKG1VYM3U0H8CK1QRnHSUxxu0XCALCsGwDJoMLKuFI3OIoR11Foozb/U6UJNAYSR4lfxl2BR64wajhAUePDT52msmssCArgVfI9p68F98AHe84rWPy7bU3D/iHa2aIPMsK/f1KLo0Ev93KciYdgXd10wAY0+hac4SVCLMnjvmwa9Naf1TQ/xne4BbZ0phQPaTLGs1JH2RKFtr5l2i+x5LpW4cfnp0AYQZeq5demZqQZ3VKDjQaF+PHvZv1Xo+IHFT4ytg/mTnfBO2R0u45t37375q2D2x9T1NFwINdqaawBVlcK34aFOZ2rgJpd4U0gbVGfgdHc5StIYx9OSltqMdj8zhHWIhpIcQRAEcWfyMJ1iqoBKjoIXinpXWxNizqmXa9V1/p6898WyZMCQ97Whl2pQEuBtrJ+iRWvxrKtJckiTvbBoMgWD4aqgsigsuBJLZzmAppj4WJYE1E8fdZfvcTjVy+P5n/hzKZnfhlZFkrY8ab8E0c+Pg9JwH8Dv3mtH6o9p28HVxrvrS7laLu5qw1XjbAXypGU9e0Zgf8FQjaWXsALxNdwQpbet+CQavirZjVmAVItVK/gb5yi5EpGS+tB/t06BcnvV8amM2Zw9jjFloGL+ENh80UKBQ+AkrHcDKbj6YvRdDyb5fReY3vmQ9QuGqZNqcqdI7m8tVrpA+/wD4+MO/BdC8l8krQegMZHA2FQnHK9horDpuiU/akzOkDuv7bEGYC7N92gpQZdyOE1l3KzQ9Fhb1EXBi4g16m0nFmBGFgbTeUEKDZEg4CLnz3LNjYADlKEra408wg5aaEqofILyUZy1UsoTHJTptLGtrW+8UEJi7U1tXHMMauLJOq7Eqi4wNGvdGhN2dXLoKUSEJqUHIvokDTse7yVoKoDuBEaJBkufJTDDDcb6AG8T6pa+ycieGVCDTkIHC1Es4sAapyPJ7PJJJmzGCFAPBvPj2YjfndyfMiId4gCGLq43e8aHXpzz40epEdvUwIu/EetluCAnKeWVqe+DlojnRFNgYMEm2HUN90pKu/iEO2cV1RdEABw5pW5wRDxGvrXUwpQYohYclaxBcyUHeGdUHkGZuwDmlqByc3mbg3II/obZHYbv7B4xXqLzpXA3D8/ew0M0VHYoYzemxL53D7gAo4oGBVHBXIJ8KyvZOFgovgQfOJsWuM0uJtcUji66kAZ85llZj/IzuN8OmdNkNNHM/QPjDioX/S7xA/I25oFZGir2L3BO7309F59oukLlnXFNbuf8uGR0x4wZSuNwYIrllAWmy8yhknPqnWrfei97/0tQdKVLowDW46HbTNCChBEyd02yr4ZNm6uFeXjEwwBZ6pecj9XJvL4ByE7H+RZMsOnbUEHv9NZ6J8aoUXu5Bb4MAt5Lcc0xOf68a/TS6RLgZ6WZe8KtaJAlYvZKb7GNDCs9JrnsXGWz3+Q1DYyZ42ZAy7NrZx8wVxDQFYseq10k2lScR5BnQ49Hoh3YPaib1N9wSZy0yqKsq04syA1E+uGFTH5kOUn82zyA3KFUdAi42ktZpJUzGirDMeVKlopDUwXmAvmY/Hk0XEYbpiiHTOdC6BUzxlhxMDqr+wpLbkx31JfFwnz7VAp3abn9jB9auhDAg526gadtTfj4cEUPXmahzN2Vma72vXUV+nxJQqWkd2dajOyna62l4QgEpBcYBBTy5Xxm3U1p+8NCOrSqDX03B1/gISydqmKHwcEx3xd+P6RZ/0bRPJYYksW/z/N3Yt+bgPJHs3c0kMSRnwnuDg65RbEG2xnl9gfZVIVYooaobFFVTdW0+Y/Q2IMj1YUN3cj55mKCsp0OfpAsBfw5f1OYz2cjibT2VB+dSeNnC8h6LLFNUqFF6J6eKhz1bYxHJcZfVKh7hqM5KXcWfDp8Lqf2Laj0zCYZUn2n6iqVO0ytUtpG/BthPQLERfb8VosYA0j6J8UVL9RSiujV8pEV+vPhUWbuThKLEnqi20snZ1f0n6ab59MS9vhLZMWrOXTJ+Nimr0kZzwChm2BwUa7FXo1EN5OyjQlDC6+EPRKCFwaw1l5NncFxgMeXcod+51t5coHt3k+lzsosMK0j3B36p/j5vG6uN58+ZOA902Ayf34aviQAZdGQQy815jZIKmXf3vNSXT/9s7eIniMQ0EN80TQEMhb+uZ8NiVuQDTRB4esP43GIPwAw6OzPALhupTms5CEKY4lNWA416q0BFnRZRwNu/l08Q39aJdREYxoAXnu2TLzxK7CE+T5Z9nnZ/rqpYwMk/7iskyt/eJk544T0qUHSbZyz9Pb++H+dD6aHW0zNjBSOtHuRS/dqBHpwUGqiF7lRT2EY4/ZlEkqebzyXJn8K4AzZEwkDHIcr9NUxfcjHFWkXyVktLnDeQWfPqQ9XUr1mouesanV+quzywvRNFwigXNPhYhxt1LFrRQTcCh0GgrFEhc7zNKYCPTOLcYzE69MS+kOJ+iiquVW0tcS80h1pWnzvf8DoYllwA=='}
EXPECTED = {'Atlas_v3_2.py': 'cd60f877d245cd2955187788e562b6fe8fcb4c448829514cbdf376b81455829e', 'Atlas_v3_2_5.py': '4411658eb868a353f72815ece14101b4c7fc135d8e7d20973e865ec98971f55e', 'Atlas_v3_2_5_preregistro.md': 'bcb9d59dc0e13859bd7c6a9d7f6ab63fc4eea2f5175f83892b8520926aff5f37'}
for name, packed in PAYLOAD.items():
    path = CODE/name
    path.write_bytes(zlib.decompress(base64.b64decode(packed)))
    assert sha(path) == EXPECTED[name], name
print('Dataset:', DATA)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(2)])
print('Scripts verificados:', EXPECTED)

import os, subprocess, sys, json, numpy as np, pandas as pd
from IPython.display import FileLink, display

def export_audit(out, code, data, dest, *, local_proxy_test=False):
    expected_mode = "proxy" if local_proxy_test else "full"
    expected_reference = "Narrow proxy" if local_proxy_test else "reconstructed v2.7"
    o = out/'O3'
    r = json.loads((o/'report.json').read_text())
    summary = json.loads((out/'report.json').read_text())
    manifest = json.loads((out/'manifest.json').read_text())
    assert r['mode']==expected_mode and r['origin']=='O3', 'Modo/origen incorrecto; no se aceptan proxies en producción'
    assert r['reference']==expected_reference, 'Referencia incorrecta'
    assert r['counts']==[1999,56,256], r['counts']
    assert r['cache']['train_max'] < r['cache']['cut']=='2024052100'
    assert r['code_sha256']==sha(code/'Atlas_v3_2_5.py')==EXPECTED['Atlas_v3_2_5.py']
    assert manifest['code']['Atlas_v3_2.py']==sha(code/'Atlas_v3_2.py')==EXPECTED['Atlas_v3_2.py']
    assert summary['preregister']==sha(code/'Atlas_v3_2_5_preregistro.md')==EXPECTED['Atlas_v3_2_5_preregistro.md']
    assert sha(o/'reference_cache.npz')==r['cache']['sha256']
    for n, digest in r['data'].items(): assert sha(data/n)==digest, n
    with np.load(o/'reference_cache.npz',allow_pickle=False) as f:
        m=json.loads(str(f['metadata']))
        families=sorted(set(f.files)-{'metadata','times'})
        assert set(families)==({'Narrow'} if local_proxy_test else {'CNN','ResNet','Mutalisk','Viper','Kernel','Clock','Regional','Narrow'})
        assert m['mode']==expected_mode and m['origin']=='O3' and m['epochs']==(None if local_proxy_test else [25,36])
        assert m['data']==r['data'] and m['patch_sha256']==r['code_sha256']
        assert len(m['train_times'])==1999 and max(m['train_times'])==m['train_max']<m['cut']
        assert m['cut']==r['cache']['cut'] and m['train_max']==r['cache']['train_max']
        assert len(m['query_times'])==312 and np.array_equal(f['times'],m['query_times'])
        assert len(set(m['train_times']))==1999 and len(set(m['query_times']))==312
        assert all(m['cut']<=t<'2024052800' and int(t[-2:])%3==0 for t in m['query_times'][:56])
        assert all('2024052800'<=t<'2024070100' and int(t[-2:])%3==0 for t in m['query_times'][56:])
    frames=[]
    def add(section, df):
        df=df.copy(); df.insert(0,'section',section); frames.append(df)
    def meta(section, obj):
        add(section,pd.DataFrame([{'field':k,'value':json.dumps(v,ensure_ascii=False,sort_keys=True)} for k,v in obj.items()]))
    if not local_proxy_test:
        assert summary['mode']=='full' and [x['origin'] for x in summary['reports']]==['O3'], 'Resumen no exclusivo O3'
    meta('report',r); meta('cache_metadata',m); meta('gate',summary['gate']); meta('manifest',manifest)
    meta('audit',{'reference_label':'Narrow_LOCAL_TEST' if local_proxy_test else 'Ultralisk27','reference_definition':expected_reference, 'primary':'MixBiasVar','retrospective':True,'code_verified':True,'families':families,'submission_authorized':False})
    for section,name in [('global','validation_metrics.csv'),('weekly','weekly_scores.csv'),('daily','daily_scores.csv'),('correlation','correlations.csv')]:
        assert sha(o/name)==manifest['files']['O3/'+name], name
        df=pd.read_csv(o/name)
        if section!='correlation': assert {'Reference','MixBiasVar'}.issubset(df['model'])
        add(section,df)
    add('weights',pd.DataFrame([{'model':name,'variable':var,'weight':w} for name,p in r['policies'].items() for var,w in zip(['RH','T','U','V'],p['weights'])]))
    assert sha(o/'daily_sse.npz')==manifest['files']['O3/daily_sse.npz']
    with np.load(o/'daily_sse.npz',allow_pickle=False) as z:
        days=z['days']; counts=z['counts']; assert len(days)==34 and sum(counts)==627200 and sum(counts>0)==32
        norm=np.array([25.360,2.0652,2.6776,2.0876]); glob=pd.read_csv(o/'validation_metrics.csv').set_index('model')
        assert {'Reference','MixBiasVar'}.issubset(z.files)
        for name in sorted(set(z.files)-{'days','counts'}):
            s=z[name]; assert s.shape==(34,4) and np.isfinite(s).all() and (s>=0).all() and (s[counts==0]==0).all()
            rmse=np.sqrt(s.sum(0)/counts.sum()); assert np.allclose(rmse,glob.loc[name,['RMSE_RH','RMSE_T','RMSE_U','RMSE_V']].to_numpy(float),rtol=0,atol=1e-8)
            assert abs(1-(rmse/norm).mean()-glob.loc[name,'skill'])<1e-9
            df=pd.DataFrame(s,columns=['SSE_RH','SSE_T','SSE_U','SSE_V']); df['date']=days; df['day']=np.arange(1,35); df['pixels_hours']=counts; df['model']=name; df['origin']='O3'; add('daily_sse',df)
    result=pd.concat(frames,ignore_index=True,sort=False)
    dest=Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp=dest.with_suffix('.tmp')
    result.to_csv(tmp,index=False,float_format='%.17g')
    check=pd.read_csv(tmp,low_memory=False)
    assert len(check)==len(result) and check['section'].value_counts().to_dict()==result['section'].value_counts().to_dict()
    tmp.replace(dest)
    assert dest.is_file() and dest.stat().st_size>0
    print('CSV verificado:', dest.stat().st_size, 'bytes; SHA256:', sha(dest))
    print('Auditoría FULL O3:',len(result),'filas de datos')
    print('NO es submission:',dest)
    return dest


DEST = Path('/kaggle/working/Atlas325_FULL_O3_AUDIT.csv')
required_outputs = [OUT/'report.json', OUT/'manifest.json'] + [OUT/'O3'/n for n in ['report.json','reference_cache.npz','validation_metrics.csv','weekly_scores.csv','daily_scores.csv','daily_sse.npz','correlations.csv']]
if all(p.is_file() for p in required_outputs):
    print('FULL O3 encontrado: se recupera sin reentrenar.')
else:
    assert torch.cuda.device_count()==2 and all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Selecciona GPU T4 x2'
    subprocess.run([sys.executable,str(CODE/'Atlas_v3_2_5.py'),'--mode','full','--origin','O3','--data',str(DATA),'--out',str(OUT)],cwd=CODE,check=True)
export_audit(OUT,CODE,DATA,DEST)
os.chdir('/kaggle/working')
print('Descarga desde el panel Files/Output de Kaggle:', DEST.name)
print('Si el enlace devuelve 404, usa el panel de archivos; el CSV ya fue leído y verificado.')
display(FileLink(DEST.name))
